# 📧 Spam Ham Classification using Bag of Words, TF-IDF and Machine Learning

## 📌 Objective

The objective of this project is to build a Machine Learning model that can automatically classify SMS messages into two categories:

- **Ham** → Legitimate SMS
- **Spam** → Unwanted or promotional SMS

The project demonstrates the complete Natural Language Processing (NLP) workflow, including:

- Text preprocessing
- Snowball Stemming
- Lemmatization
- Feature Extraction using Bag of Words
- Feature Extraction using TF-IDF
- Machine Learning Model Training
- Performance Evaluation

Finally, we compare the classification performance of:

- Snowball Stemming vs Lemmatization
- Bag of Words vs TF-IDF

to determine which preprocessing technique provides better classification accuracy.

# 📦 Import Required Libraries

In this section, we import all the Python libraries required for:

- Data manipulation
- Text preprocessing
- Feature extraction
- Machine Learning
- Model evaluation

In [1]:
# Import Regular Expression Library
import re

# Import NumPy
import numpy as np

# Import Pandas
import pandas as pd

# Import Natural Language Toolkit
import nltk

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# Import Stopwords
from nltk.corpus import stopwords

# Import Snowball Stemmer
from nltk.stem import SnowballStemmer

# Import WordNet Lemmatizer
from nltk.stem import WordNetLemmatizer

# Import Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

# Import TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Import Train-Test Split
from sklearn.model_selection import train_test_split

# Import Naive Bayes Classifier
from sklearn.naive_bayes import MultinomialNB

# Import Accuracy Metric
from sklearn.metrics import accuracy_score

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/milind/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/milind/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/milind/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


# 📂 Load the Dataset

The SMS Spam Collection dataset consists of SMS messages labeled as:

- **Ham** → Legitimate Message
- **Spam** → Unwanted Message

Each record contains:

- **label**
- **message**

In [2]:
# Load the SMS Spam Collection Dataset
messages = pd.read_csv(
    "SMSSpamCollection.txt",
    sep="\t",
    names=["label", "message"]
)

# Display the first five records
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# 🔍 Exploratory Data Analysis

Before preprocessing the text, let us understand the dataset.

We will check:

- Dataset Shape
- Missing Values
- Class Distribution

In [3]:
# Display Dataset Shape
print("Dataset Shape:", messages.shape)

# Display Missing Values
print("\nMissing Values")
print(messages.isnull().sum())

# Display Class Distribution
print("\nClass Distribution")
print(messages["label"].value_counts())

Dataset Shape: (5572, 2)

Missing Values
label      0
message    0
dtype: int64

Class Distribution
label
ham     4825
spam     747
Name: count, dtype: int64


# 🧹 Text Preprocessing

Machine Learning algorithms cannot understand raw text directly.

Therefore, the SMS messages need to be cleaned and converted into a numerical representation.

In this project, we will compare **two preprocessing techniques**:

1. **Snowball Stemming**
2. **WordNet Lemmatization**

Both techniques perform the following operations:

- Remove special characters
- Convert text to lowercase
- Tokenize the text into words
- Remove English stopwords
- Apply stemming or lemmatization
- Join the processed words back into a sentence

Finally, we will compare the performance of both preprocessing techniques using **Bag of Words** and **TF-IDF**.

In [5]:
# Store English stopwords
english_stopwords = set(stopwords.words("english"))

# Initialize Snowball Stemmer
stemmer = SnowballStemmer("english")

# Initialize WordNet Lemmatizer
lemmatizer = WordNetLemmatizer()

# 🌿 Snowball Stemming

**Stemming** reduces words to their root form by removing prefixes and suffixes.

In this project, we use the **Snowball Stemmer**, which is generally more accurate than the Porter Stemmer for English text.

### Examples

| Original Word | Stemmed Word |
|--------------|--------------|
| playing | play |
| running | run |
| studies | studi |
| loving | love |

Although stemming is fast, the resulting words are not always valid dictionary words.

In [6]:
def preprocess_with_stemming(text):
    """
    Clean and preprocess text using Snowball Stemming.

    Parameters:
        text (str): Raw SMS message

    Returns:
        str: Cleaned and stemmed SMS message
    """

    # Remove all characters except alphabets
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Convert text to lowercase
    text = text.lower()

    # Split sentence into words
    words = text.split()

    # Remove stopwords and apply stemming
    processed_words = [
        stemmer.stem(word)
        for word in words
        if word not in english_stopwords
    ]

    # Join words back into a sentence
    return " ".join(processed_words)

# 📚 Create the Stemmed Corpus

Now, the preprocessing function is applied to every SMS message in the dataset.

The resulting collection of cleaned and stemmed messages is called the **Stemmed Corpus**.

In [7]:
# Apply Snowball Stemming to every SMS message
stemmed_corpus = [
    preprocess_with_stemming(message)
    for message in messages["message"]
]

print("Total Messages:", len(stemmed_corpus))

Total Messages: 5572


In [8]:
# Compare Original and Stemmed Message
index = 0

print("Original Message")
print("-" * 60)
print(messages["message"].iloc[index])

print("\nStemmed Message")
print("-" * 60)
print(stemmed_corpus[index])

Original Message
------------------------------------------------------------
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Stemmed Message
------------------------------------------------------------
go jurong point crazi avail bugi n great world la e buffet cine got amor wat


# 🍃 WordNet Lemmatization

**Lemmatization** converts a word into its meaningful base form (called the *lemma*).

Unlike stemming, lemmatization produces valid dictionary words.

### Examples

| Original Word | Lemmatized Word |
|--------------|-----------------|
| playing | playing |
| running | running |
| studies | study |
| children | child |
| cars | car |

Lemmatization generally preserves the meaning of words better than stemming, but it is computationally slower.

In [10]:
def preprocess_with_lemmatization(text):
    """
    Clean and preprocess text using WordNet Lemmatization.

    Parameters:
        text (str): Raw SMS message

    Returns:
        str: Cleaned and lemmatized SMS message
    """

    # Remove all characters except alphabets
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Convert text to lowercase
    text = text.lower()

    # Split sentence into words
    words = text.split()

    # Remove stopwords and apply lemmatization
    processed_words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in english_stopwords
    ]

    # Join words back into a sentence
    return " ".join(processed_words)

# 📚 Create the Lemmatized Corpus

The preprocessing function is now applied to every SMS message to create a **Lemmatized Corpus**.

This corpus will later be used to compare its performance against the Stemmed Corpus.

In [11]:
# Apply Lemmatization to every SMS message
lemmatized_corpus = [
    preprocess_with_lemmatization(message)
    for message in messages["message"]
]

print("Total Messages:", len(lemmatized_corpus))

Total Messages: 5572


In [12]:
# Compare Original and Lemmatized Message
index = 0

print("Original Message")
print("-" * 60)
print(messages["message"].iloc[index])

print("\nLemmatized Message")
print("-" * 60)
print(lemmatized_corpus[index])

Original Message
------------------------------------------------------------
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Lemmatized Message
------------------------------------------------------------
go jurong point crazy available bugis n great world la e buffet cine got amore wat


In [13]:
# Compare all preprocessing techniques
index = 5

print("Original Message")
print("-" * 60)
print(messages["message"].iloc[index])

print("\nSnowball Stemmed")
print("-" * 60)
print(stemmed_corpus[index])

print("\nLemmatized")
print("-" * 60)
print(lemmatized_corpus[index])

Original Message
------------------------------------------------------------
FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv

Snowball Stemmed
------------------------------------------------------------
freemsg hey darl week word back like fun still tb ok xxx std chgs send rcv

Lemmatized
------------------------------------------------------------
freemsg hey darling week word back like fun still tb ok xxx std chgs send rcv


# 🎯 Prepare the Target Variable

Machine Learning models require the target variable to be in numerical format.

The dataset contains two classes:

- **ham** → Legitimate SMS
- **spam** → Unwanted SMS

We convert these labels into binary values using `pd.get_dummies()`.

| Label | Encoded Value |
|-------|--------------:|
| ham | 0 |
| spam | 1 |

In [14]:
# Convert labels into numerical values
y = pd.get_dummies(messages["label"], drop_first=True)

# Convert DataFrame to NumPy array
y = y.values.ravel()

print("Target Variable Shape:", y.shape)

# Display first 10 target values
print("\nFirst 10 Labels:")
print(y[:10])

Target Variable Shape: (5572,)

First 10 Labels:
[False False  True False False  True False False  True  True]


# ✂️ Train-Test Split

The dataset is divided into two parts:

- **Training Set (80%)** → Used to train the Machine Learning model.
- **Testing Set (20%)** → Used to evaluate the model on unseen data.

This helps us measure how well the model generalizes to new SMS messages.

# 🤖 Model Evaluation Function

Instead of writing the same code repeatedly for each experiment, we create a reusable function.

The function will:

1. Split the dataset into training and testing sets.
2. Convert text into numerical features using either:
   - Bag of Words
   - TF-IDF
3. Train a **Multinomial Naive Bayes** classifier.
4. Make predictions on the test set.
5. Calculate the classification accuracy.

This approach keeps the notebook clean and avoids duplicate code.

In [16]:
# Import required libraries
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [17]:
def evaluate_model(corpus, vectorizer):
    """
    Train and evaluate a Multinomial Naive Bayes model.

    Parameters:
        corpus (list): Preprocessed text corpus.
        vectorizer: CountVectorizer or TfidfVectorizer.

    Returns:
        float: Model accuracy.
    """

    # Split the dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        corpus,
        y,
        test_size=0.20,
        random_state=42
    )

    # Learn the vocabulary from training data
    X_train = vectorizer.fit_transform(X_train)

    # Transform the testing data
    X_test = vectorizer.transform(X_test)

    # Initialize the classifier
    model = MultinomialNB()

    # Train the model
    model.fit(X_train, y_train)

    # Predict labels for the test data
    predictions = model.predict(X_test)

    # Calculate model accuracy
    accuracy = accuracy_score(y_test, predictions)

    return accuracy

# 📦 Experiment 1 — Snowball Stemming + Bag of Words

In this experiment:

- **Preprocessing:** Snowball Stemming
- **Feature Extraction:** Bag of Words
- **Classifier:** Multinomial Naive Bayes

The SMS messages are converted into a Bag of Words representation before training the classifier.

In [18]:
# Create Bag of Words vectorizer
bow_vectorizer = CountVectorizer(max_features=5000)

# Train and evaluate the model
stem_bow_accuracy = evaluate_model(
    stemmed_corpus,
    bow_vectorizer
)

print(f"Snowball + BoW Accuracy : {stem_bow_accuracy:.4f}")

Snowball + BoW Accuracy : 0.9839


# 📦 Experiment 2 — Lemmatization + Bag of Words

In this experiment:

- **Preprocessing:** WordNet Lemmatization
- **Feature Extraction:** Bag of Words
- **Classifier:** Multinomial Naive Bayes

The performance of the model is compared with Snowball Stemming using the same feature extraction technique.

In [19]:
# Create Bag of Words vectorizer
bow_vectorizer = CountVectorizer(max_features=5000)

# Train and evaluate the model
lemma_bow_accuracy = evaluate_model(
    lemmatized_corpus,
    bow_vectorizer
)

print(f"Lemmatization + BoW Accuracy : {lemma_bow_accuracy:.4f}")

Lemmatization + BoW Accuracy : 0.9848


# 📦 Experiment 3 — Snowball Stemming + TF-IDF

In this experiment:

- **Preprocessing:** Snowball Stemming
- **Feature Extraction:** TF-IDF
- **Classifier:** Multinomial Naive Bayes

TF-IDF assigns higher importance to informative words and lower importance to commonly occurring words.

In [21]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Train and evaluate the model
stem_tfidf_accuracy = evaluate_model(
    stemmed_corpus,
    tfidf_vectorizer
)

print(f"Snowball + TF-IDF Accuracy : {stem_tfidf_accuracy:.4f}")

Snowball + TF-IDF Accuracy : 0.9731


# 📦 Experiment 4 — Lemmatization + TF-IDF

In this experiment:

- **Preprocessing:** WordNet Lemmatization
- **Feature Extraction:** TF-IDF
- **Classifier:** Multinomial Naive Bayes

This is the final experiment and allows us to compare all four preprocessing-feature extraction combinations.

In [22]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Train and evaluate the model
lemma_tfidf_accuracy = evaluate_model(
    lemmatized_corpus,
    tfidf_vectorizer
)

print(f"Lemmatization + TF-IDF Accuracy : {lemma_tfidf_accuracy:.4f}")

Lemmatization + TF-IDF Accuracy : 0.9713


# 📊 Accuracy Comparison

The following table summarizes the performance of all four experiments.

This comparison helps us identify:

- Which preprocessing technique performs better.
- Whether Bag of Words or TF-IDF provides better classification accuracy.

In [23]:
# Create a comparison table
comparison = pd.DataFrame({
    "Preprocessing": [
        "Snowball",
        "Lemmatization",
        "Snowball",
        "Lemmatization"
    ],
    "Vectorizer": [
        "Bag of Words",
        "Bag of Words",
        "TF-IDF",
        "TF-IDF"
    ],
    "Accuracy": [
        stem_bow_accuracy,
        lemma_bow_accuracy,
        stem_tfidf_accuracy,
        lemma_tfidf_accuracy
    ]
})

# Display the comparison table
comparison

,Preprocessing,Vectorizer,Accuracy
0,Snowball,Bag of Words,0.983857
1,Lemmatization,Bag of Words,0.984753
2,Snowball,TF-IDF,0.973094
3,Lemmatization,TF-IDF,0.971300


In [24]:
# Identify the best-performing model
best_model = comparison.loc[comparison["Accuracy"].idxmax()]

print("🏆 Best Performing Model")
print("-" * 40)
print(f"Preprocessing : {best_model['Preprocessing']}")
print(f"Vectorizer    : {best_model['Vectorizer']}")
print(f"Accuracy      : {best_model['Accuracy']:.4f}")

🏆 Best Performing Model
----------------------------------------
Preprocessing : Lemmatization
Vectorizer    : Bag of Words
Accuracy      : 0.9848


# ✅ Conclusion

In this project, we built an SMS Spam Classification system using **Natural Language Processing (NLP)** and **Machine Learning**.

We compared two preprocessing techniques:

- Snowball Stemming
- WordNet Lemmatization

and two feature extraction techniques:

- Bag of Words (BoW)
- TF-IDF

Each combination was evaluated using a **Multinomial Naive Bayes** classifier.

The comparison table helped identify the best-performing approach for this dataset.

This project demonstrates a complete NLP workflow, including:

- Data Loading
- Exploratory Data Analysis
- Text Preprocessing
- Feature Extraction
- Model Training
- Model Evaluation
- Performance Comparison